In [1]:
import re
import json
import time
import random
import sys
from datetime import date
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
DATE_START = date(2026, 1, 1)
DATE_END   = date(2026, 3, 31)

MAX_PAGES     = 3
FETCH_CONTENT = True
DELAY_MIN     = 1.5   # detik minimum antar request
DELAY_MAX     = 3.0   # detik maksimum antar request

OUTPUT_DIR  = Path("C:/content")
OUTPUT_CSV  = OUTPUT_DIR / "geo_text_malang.csv"
OUTPUT_JSON = OUTPUT_DIR / "geo_text_malang.json"
OUTPUT_GEO  = OUTPUT_DIR / "geo_text_malang.geojson"

In [3]:
"""
Geo-Text Mining – Kabupaten Malang
===================================
Scraping berita kejadian (kriminalitas, bencana, kesehatan, konflik sosial)
di 33 kecamatan Kabupaten Malang, periode 1 Jan – 31 Mar 2026.

Output : CSV, JSON, GeoJSON (siap peta per kecamatan)
å
VERSI: requests + httpx (tanpa Playwright / browser headless)
"""



# ─────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────



# ─────────────────────────────────────────────
#  SESSION SETUP (rotate user-agent, headers wajar)
# ─────────────────────────────────────────────

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) "
    "Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

session = requests.Session()
session.headers.update({
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection":      "keep-alive",
    "DNT":             "1",
})


def get_html(url: str, timeout: int = 20) -> str:
    """Ambil HTML satu halaman dengan requests. Return string kosong jika gagal."""
    session.headers["User-Agent"] = random.choice(USER_AGENTS)
    try:
        resp = session.get(url, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        resp.encoding = resp.apparent_encoding or "utf-8"
        return resp.text
    except requests.exceptions.HTTPError as e:
        print(f"    [HTTP {e.response.status_code}] {url[:70]}")
        return ""
    except requests.exceptions.ConnectionError:
        print(f"    [CONN ERR] {url[:70]}")
        return ""
    except requests.exceptions.Timeout:
        print(f"    [TIMEOUT] {url[:70]}")
        return ""
    except Exception as e:
        print(f"    [ERR] {e}")
        return ""


def smart_delay():
    """Jeda acak agar tidak terdeteksi sebagai bot."""
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))


# ─────────────────────────────────────────────
#  33 KECAMATAN KABUPATEN MALANG + KOORDINAT CENTROID
# ─────────────────────────────────────────────

KECAMATAN = {
    "Ampelgading":         (-8.2583, 112.7667),
    "Bantur":              (-8.3167, 112.5167),
    "Bululawang":          (-8.1333, 112.6833),
    "Dampit":              (-8.2333, 112.7833),
    "Dau":                 (-7.9833, 112.5667),
    "Donomulyo":           (-8.2833, 112.4167),
    "Gedangan":            (-8.3333, 112.6667),
    "Gondanglegi":         (-8.1667, 112.6333),
    "Jabung":              (-7.9667, 112.7167),
    "Kalipare":            (-8.2167, 112.4667),
    "Karangploso":         (-7.9333, 112.6167),
    "Kasembon":            (-7.8167, 112.3667),
    "Kepanjen":            (-8.1333, 112.5667),
    "Kromengan":           (-8.1000, 112.5167),
    "Lawang":              (-7.8333, 112.6833),
    "Ngajum":              (-8.0833, 112.5000),
    "Ngantang":            (-7.8833, 112.3833),
    "Pagak":               (-8.2000, 112.5167),
    "Pagelaran":           (-8.1500, 112.5833),
    "Pakis":               (-7.9833, 112.7000),
    "Pakisaji":            (-8.1000, 112.6000),
    "Poncokusumo":         (-8.0167, 112.7667),
    "Pujon":               (-7.9000, 112.4167),
    "Singosari":           (-7.9000, 112.6667),
    "Sumbermanjing Wetan": (-8.3667, 112.6833),
    "Sumberpucung":        (-8.1667, 112.5333),
    "Tajinan":             (-8.1000, 112.6667),
    "Tirtoyudo":           (-8.2833, 112.7167),
    "Tumpang":             (-8.0000, 112.7333),
    "Turen":               (-8.1667, 112.6833),
    "Wagir":               (-8.0333, 112.5500),
    "Wajak":               (-8.0833, 112.7167),
    "Wonosari":            (-8.1500, 112.5167),
}

KECAMATAN_ALIASES: dict[str, str] = {
    "sumbermanjing":      "Sumbermanjing Wetan",
    "sumbermanjingwetan": "Sumbermanjing Wetan",
    "smw":                "Sumbermanjing Wetan",
    "kpj":                "Kepanjen",
}

# ─────────────────────────────────────────────
#  KATEGORI & KEYWORD
# ─────────────────────────────────────────────

CATEGORIES: dict[str, list[str]] = {
    "kriminalitas": [
        "pencurian", "maling", "begal", "narkoba", "sabu", "ganja",
        "pil koplo", "pengedar", "kekerasan", "penganiayaan",
        "pembunuhan", "perampokan", "perkosaan", "pemerkosaan",
        "penipuan", "perampasan", "curanmor", "jambret",
    ],
    "bencana": [
        "longsor", "tanah longsor", "banjir", "banjir bandang",
        "kebakaran", "rumah terbakar", "gedung terbakar",
        "gempa", "gempa bumi", "puting beliung", "angin kencang",
        "badai", "bencana alam", "pohon tumbang", "abrasi",
    ],
    "kesehatan": [
        "keracunan", "keracunan makanan", "keracunan massal",
        "wabah", "DBD", "demam berdarah", "diare massal",
        "penyakit menular", "KLB", "kejadian luar biasa",
    ],
    "konflik_sosial": [
        "demo", "demonstrasi", "unjuk rasa", "aksi massa",
        "konflik warga", "bentrok", "perkelahian massal",
        "sengketa lahan", "sengketa tanah", "kerusuhan",
        "tawuran", "pemblokiran jalan",
    ],
}

# ─────────────────────────────────────────────
#  SUMBER BERITA
# ─────────────────────────────────────────────

SOURCES = [
    {
        "name":       "Surya Malang",
        "search_url": "https://suryamalang.tribunnews.com/search?q={query}&page={page}",
        "article_re": re.compile(r"suryamalang\.tribunnews\.com/\d{4}/\d{2}/\d{2}/"),
        "article_sel": ["li.lsi", "div.lsi", "li.list--item", "article"],
    },
    {
        "name":       "Detik Jatim",
        "search_url": "https://www.detik.com/jatim/search/?query={query}&sortby=time&page={page}",
        "article_re": re.compile(r"detik\.com/jatim/[^/]+/d-\d+"),
        "article_sel": ["article.list-content__item", "article.media", "article"],
    },
    {
        "name":       "Antara Jatim",
        "search_url": (
            "https://www.antaranews.com/search?q={query}"
            "&from=2026-01-01&to=2026-03-31&page={page}"
        ),
        "article_re": re.compile(r"antaranews\.com/berita/\d+/"),
        "article_sel": ["div.simple-list li", "ul.simple-list li", "article", "li"],
    },
    {
        "name":       "Radar Malang",
        "search_url": "https://radarmalang.jawapos.com/?s={query}&paged={page}",
        "article_re": re.compile(r"radarmalang\.jawapos\.com/[^/]+/\d{4}/\d{2}/\d{2}/"),
        "article_sel": ["article.jeg_post", "article.mvp-blog-story-wrap", "article"],
    },
    {
        "name":       "Malang Times",
        "search_url": "https://www.malangtimes.com/?s={query}&paged={page}",
        "article_re": re.compile(r"malangtimes\.com/baca/\d+/"),
        "article_sel": ["article", "div.post", "div[class*='jeg_post']"],
    },
    {
        "name":       "IDN Times Jatim",
        "search_url": "https://www.idntimes.com/search?q={query}&page={page}",
        "article_re": re.compile(r"idntimes\.com/[^/]+/[^/]+/[^/]+-\d+"),
        "article_sel": ["div[class*='card']", "article", "li[class*='search']"],
    },
]

CONTENT_SELECTORS = [
    "div.detail__body-text", "div[class*='detail-text']",
    "div.content-detail",    "div.article__content",
    "div[class*='article-body']", "div.content-body",
    "div.entry-content",     "div.post-content",
    "div[itemprop='articleBody']", "article",
]

DATE_SELECTORS = [
    "time[datetime]",          "span[class*='date']",
    "span[class*='time']",     "meta[property='article:published_time']",
    "div[class*='date']",
]

# ─────────────────────────────────────────────
#  DATE PARSING
# ─────────────────────────────────────────────

_MONTH_ID = {
    "januari":1, "februari":2, "maret":3,     "april":4,
    "mei":5,     "juni":6,     "juli":7,       "agustus":8,
    "september":9,"oktober":10,"november":11, "desember":12,
    "jan":1,"feb":2,"mar":3,"apr":4,"jun":6,"jul":7,
    "agu":8,"agt":8,"sep":9,"okt":10,"nov":11,"des":12,
}

_URL_DATE_RE = re.compile(r"/(\d{4})/(\d{2})/(\d{2})[/\-]")


def parse_date(raw: str) -> date | None:
    if not raw:
        return None
    raw = raw.strip().lower()
    m = re.match(r"(\d{4})-(\d{2})-(\d{2})", raw)
    if m:
        try: return date(int(m[1]), int(m[2]), int(m[3]))
        except: pass
    m = re.search(r"(\d{1,2})\s+(\w+)\s+(\d{4})", raw)
    if m:
        mo = _MONTH_ID.get(m[2])
        if mo:
            try: return date(int(m[3]), mo, int(m[1]))
            except: pass
    m = re.match(r"(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})", raw)
    if m:
        try: return date(int(m[3]), int(m[2]), int(m[1]))
        except: pass
    return None


def date_from_url(url: str) -> date | None:
    m = _URL_DATE_RE.search(url)
    if m:
        try: return date(int(m[1]), int(m[2]), int(m[3]))
        except: pass
    return None


def in_date_range(dt: date | None) -> bool:
    if dt is None:
        return True
    return DATE_START <= dt <= DATE_END


# ─────────────────────────────────────────────
#  KECAMATAN & CATEGORY DETECTION
# ─────────────────────────────────────────────

_KEC_PATTERNS: list[tuple[str, re.Pattern]] = []
for _k in KECAMATAN:
    _variants = [_k.lower(), _k.lower().replace(" ", "")]
    _pat = "|".join(re.escape(v) for v in set(_variants))
    _KEC_PATTERNS.append((_k, re.compile(r"\b(?:" + _pat + r")\b", re.IGNORECASE)))

_ALIAS_PAT: list[tuple[str, re.Pattern]] = [
    (canonical, re.compile(r"\b" + re.escape(alias) + r"\b", re.IGNORECASE))
    for alias, canonical in KECAMATAN_ALIASES.items()
]

_CAT_PATTERNS: dict[str, re.Pattern] = {
    cat: re.compile(
        r"\b(?:" + "|".join(re.escape(kw) for kw in kws) + r")\b", re.IGNORECASE
    )
    for cat, kws in CATEGORIES.items()
}


def detect_kecamatan(text: str) -> list[str]:
    found = set()
    for name, pat in _KEC_PATTERNS:
        if pat.search(text):
            found.add(name)
    for canonical, pat in _ALIAS_PAT:
        if pat.search(text) and canonical in KECAMATAN:
            found.add(canonical)
    return sorted(found)


def detect_categories(text: str) -> list[str]:
    return [cat for cat, pat in _CAT_PATTERNS.items() if pat.search(text)]


def detect_keywords_matched(text: str) -> list[str]:
    matched = []
    for kws in CATEGORIES.values():
        for kw in kws:
            if re.search(r"\b" + re.escape(kw) + r"\b", text, re.IGNORECASE):
                matched.append(kw)
    return list(set(matched))


# ─────────────────────────────────────────────
#  QUERY BUILDER
# ─────────────────────────────────────────────

def build_queries() -> list[dict]:
    queries = []

    for cat, kws in CATEGORIES.items():
        for kw in kws[:3]:
            queries.append({"query": f"{kw} kabupaten malang", "category_hint": cat})

    priority_kws = [
        "banjir", "longsor", "kebakaran", "begal", "narkoba",
        "keracunan", "demo", "konflik warga", "gempa", "puting beliung",
    ]
    for kw in priority_kws:
        queries.append({"query": f"{kw} malang 2026", "category_hint": None})

    priority_kec = [
        "Singosari", "Lawang", "Kepanjen", "Pakis", "Karangploso",
        "Dampit", "Turen", "Gondanglegi", "Bululawang",
    ]
    for kec in priority_kec:
        for cat, kws in CATEGORIES.items():
            queries.append({"query": f"{kws[0]} {kec} malang", "category_hint": cat})

    seen, unique = set(), []
    for q in queries:
        key = q["query"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(q)
    return unique


# ─────────────────────────────────────────────
#  PARSER
# ─────────────────────────────────────────────

def parse_article_list(html: str, source: dict, query_info: dict) -> list[dict]:
    if not html:
        return []
    soup = BeautifulSoup(html, "html.parser")
    items: list[dict] = []
    seen_urls: set[str] = set()
    article_re = source["article_re"]

    article_els = []
    for sel in source.get("article_sel", []):
        article_els = soup.select(sel)
        if article_els:
            break

    if not article_els:
        for sel in ["article", "div.item", "li.item", "div[class*='list-item']"]:
            article_els = soup.select(sel)
            if article_els:
                break

    if article_els:
        for art in article_els:
            title, url = "", ""
            for a in art.find_all("a", href=True):
                href = a["href"]
                full = href if href.startswith("http") else ""
                if not full or not article_re.search(full) or full in seen_urls:
                    continue
                t = a.get_text(strip=True)
                if len(t) > 20:
                    title, url = t, full
                    break
                elif not url:
                    url = full

            if not url:
                continue
            if not title:
                for ht in ["h1", "h2", "h3", "h4"]:
                    hn = art.find(ht)
                    if hn and len(hn.get_text(strip=True)) > 15:
                        title = hn.get_text(strip=True)
                        break
            if not title or url in seen_urls:
                continue
            seen_urls.add(url)

            raw_date = ""
            for ds in DATE_SELECTORS:
                dt_tag = art.select_one(ds)
                if dt_tag:
                    raw_date = (
                        dt_tag.get("datetime", "") or
                        dt_tag.get("content", "") or
                        dt_tag.get_text(strip=True)
                    )
                    if raw_date:
                        break

            items.append(_make_record(title, url, raw_date, source["name"], query_info))

    # super-fallback: scan semua link
    if not items:
        for a in soup.find_all("a", href=True):
            href  = a["href"]
            title = a.get_text(strip=True)
            full  = href if href.startswith("http") else ""
            if not full or not article_re.search(full):
                continue
            if len(title) < 15 or full in seen_urls:
                continue
            seen_urls.add(full)
            items.append(_make_record(title, full, "", source["name"], query_info))

    return items


def _make_record(title, url, raw_date, source_name, query_info) -> dict:
    parsed_dt = parse_date(raw_date)
    if parsed_dt is None:
        parsed_dt = date_from_url(url)
        if parsed_dt and not raw_date:
            raw_date = parsed_dt.isoformat()
    return {
        "source":         source_name,
        "query":          query_info["query"],
        "category_hint":  query_info.get("category_hint", ""),
        "title":          title,
        "url":            url,
        "raw_date":       raw_date,
        "date":           parsed_dt.isoformat() if parsed_dt else "",
        "in_range":       in_date_range(parsed_dt),
        "categories":     [],
        "kecamatan":      [],
        "keywords_found": [],
        "content":        "",
    }


def extract_content(html: str) -> str:
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "aside", "iframe"]):
        tag.decompose()
    body = None
    for sel in CONTENT_SELECTORS:
        body = soup.select_one(sel)
        if body:
            break
    if not body:
        body = soup.select_one("article") or soup.find("body")
    paragraphs = [
        p.get_text(strip=True)
        for p in (body or soup).find_all("p")
        if len(p.get_text(strip=True)) > 30
    ]
    text = "\n".join(paragraphs)
    return re.sub(r"\n{3,}", "\n\n", text)[:6000]


def extract_date_from_content(html: str) -> str:
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for prop in ["article:published_time", "datePublished", "DC.date"]:
        m = soup.find("meta", {"property": prop}) or soup.find("meta", {"name": prop})
        if m and m.get("content"):
            return m["content"]
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            d = json.loads(script.string or "")
            if isinstance(d, dict):
                for key in ["datePublished", "dateCreated", "uploadDate"]:
                    if d.get(key):
                        return d[key]
        except:
            pass
    text = soup.get_text(" ", strip=True)
    m = re.search(r"\d{1,2}\s+\w+\s+202[56]", text)
    return m.group(0) if m else ""


def enrich_record(rec: dict, content_html: str) -> dict:
    content = extract_content(content_html)
    rec["content"] = content

    if not rec["date"]:
        raw = extract_date_from_content(content_html)
        dt  = parse_date(raw)
        rec["raw_date"] = raw
        rec["date"]     = dt.isoformat() if dt else ""
        rec["in_range"] = in_date_range(dt)

    full_text             = rec["title"] + " " + content
    rec["categories"]     = detect_categories(full_text)
    rec["kecamatan"]      = detect_kecamatan(full_text)
    rec["keywords_found"] = detect_keywords_matched(full_text)
    return rec


# ─────────────────────────────────────────────
#  GEOJSON & AGGREGASI
# ─────────────────────────────────────────────

def build_geojson(df: pd.DataFrame) -> dict:
    features = []
    for _, row in df.iterrows():
        kecs = (
            row["kecamatan"] if isinstance(row["kecamatan"], list)
            else json.loads(row["kecamatan"]) if row["kecamatan"] else []
        )
        for kec in kecs:
            if kec not in KECAMATAN:
                continue
            lat, lon = KECAMATAN[kec]
            features.append({
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [lon, lat]},
                "properties": {
                    "kecamatan":      kec,
                    "title":          row.get("title", ""),
                    "date":           row.get("date", ""),
                    "categories":     row.get("categories", ""),
                    "keywords_found": row.get("keywords_found", ""),
                    "source":         row.get("source", ""),
                    "url":            row.get("url", ""),
                },
            })
    return {"type": "FeatureCollection", "features": features}


def aggregate_per_kecamatan(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for kec, (lat, lon) in KECAMATAN.items():
        mask = df["kecamatan"].apply(
            lambda v: kec in (
                v if isinstance(v, list)
                else json.loads(v) if v else []
            )
        )
        sub = df[mask]
        row = {"kecamatan": kec, "lat": lat, "lon": lon, "total": len(sub)}
        for cat in CATEGORIES:
            row[cat] = sub["categories"].apply(
                lambda v: cat in (
                    v if isinstance(v, list)
                    else json.loads(v) if v else []
                )
            ).sum()
        rows.append(row)
    return pd.DataFrame(rows).sort_values("total", ascending=False)


# ─────────────────────────────────────────────
#  MAIN SCRAPE (synchronous — tanpa asyncio/Playwright)
# ─────────────────────────────────────────────

def scrape_all():
    queries          = build_queries()
    all_recs         = []
    global_seen_urls: set[str] = set()

    print(f"Total query: {len(queries)} | Sumber: {len(SOURCES)}")
    print(f"Periode: {DATE_START} s.d. {DATE_END}\n")

    for q_info in queries:
        query = q_info["query"]
        print(f"\n🔍 Query: '{query}'")
        new_candidates: list[dict] = []

        for src in SOURCES:
            for pg in range(1, MAX_PAGES + 1):
                url = src["search_url"].format(
                    query=query.replace(" ", "+"), page=pg
                )
                print(f"  [{src['name']}] p{pg}: {url[:72]}")
                html = get_html(url)
                arts = parse_article_list(html, src, q_info)

                arts_new = [a for a in arts if a["url"] not in global_seen_urls]
                arts_new = [a for a in arts_new if a["in_range"] or not a["date"]]

                if not arts_new:
                    reason = "semua duplikat" if arts else "tidak ada artikel"
                    print(f"    ⚠ {reason}, lanjut halaman berikutnya.")
                    break

                for a in arts_new:
                    global_seen_urls.add(a["url"])

                print(f"    ✓ {len(arts_new)} artikel baru "
                      f"(total dikenal: {len(global_seen_urls)})")
                new_candidates.extend(arts_new)
                smart_delay()

        # Fetch konten artikel
        if FETCH_CONTENT and new_candidates:
            print(f"  → Fetch konten: {len(new_candidates)} artikel")
            for i, rec in enumerate(new_candidates, 1):
                print(f"  [{i}/{len(new_candidates)}] {rec['url'][:65]}")
                c_html = get_html(rec["url"])
                enrich_record(rec, c_html)
                smart_delay()

            new_candidates = [r for r in new_candidates if r["in_range"] or not r["date"]]
            new_candidates = [r for r in new_candidates if r["categories"] or r["category_hint"]]
            all_recs.extend(new_candidates)
        elif new_candidates:
            all_recs.extend(new_candidates)

    # Final dedup
    seen, unique = set(), []
    for r in all_recs:
        if r["url"] not in seen:
            seen.add(r["url"])
            unique.append(r)

    print(f"\n✅ Total artikel unik: {len(unique)}")

    df = pd.DataFrame(unique)
    if df.empty:
        print("⚠ Tidak ada artikel yang berhasil dikumpulkan.")
        return df, pd.DataFrame()

    for col in ["categories", "kecamatan", "keywords_found"]:
        df[col] = df[col].apply(json.dumps)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"📄 CSV     : {OUTPUT_CSV}")

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(unique, f, ensure_ascii=False, indent=2)
    print(f"📄 JSON    : {OUTPUT_JSON}")

    df_geo = df.copy()
    for col in ["categories", "kecamatan", "keywords_found"]:
        df_geo[col] = df_geo[col].apply(lambda v: json.loads(v) if v else [])

    geojson = build_geojson(df_geo)
    with open(OUTPUT_GEO, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)
    print(f"📄 GeoJSON : {OUTPUT_GEO}")

    df_agg = aggregate_per_kecamatan(df_geo)
    agg_path = OUTPUT_DIR / "geo_text_malang_summary.csv"
    df_agg.to_csv(agg_path, index=False, encoding="utf-8-sig")
    print(f"📄 Summary : {agg_path}")

    return df, df_agg


# ─────────────────────────────────────────────
#  STATISTIK
# ─────────────────────────────────────────────

def print_stats(df: pd.DataFrame, df_agg: pd.DataFrame):
    print("\n" + "=" * 55)
    print("STATISTIK HASIL GEO-TEXT MINING")
    print("=" * 55)

    print("\n📊 Artikel per kategori:")
    for cat in CATEGORIES:
        n = df["categories"].apply(
            lambda v: cat in (json.loads(v) if isinstance(v, str) else v)
        ).sum()
        print(f"  {cat:<20}: {n} artikel")

    print("\n📍 Top 10 kecamatan tersering disebut:")
    top10 = df_agg.head(10)[["kecamatan", "total"] + list(CATEGORIES.keys())]
    print(top10.to_string(index=False))

    dates = pd.to_datetime(
        df["date"].replace("", pd.NA), errors="coerce"
    ).dropna()
    if len(dates):
        print(f"\n📅 Rentang tanggal: {dates.min().date()} s.d. {dates.max().date()}")
        print("   Distribusi per bulan:")
        print(dates.dt.to_period("M").value_counts().sort_index().to_string())

    print(f"\n🗺 Total kecamatan teridentifikasi: "
          f"{(df_agg['total'] > 0).sum()} dari {len(KECAMATAN)}")


# ─────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────

def main():
    print("=" * 55)
    print("GEO-TEXT MINING – KABUPATEN MALANG")
    print(f"Periode : {DATE_START} s.d. {DATE_END}")
    print(f"Sumber  : {len(SOURCES)} portal berita")
    print(f"Output  : {OUTPUT_DIR.resolve()}")
    print("=" * 55)

    df, df_agg = scrape_all()

    if not df.empty:
        print_stats(df, df_agg)
        print("\n─── Preview 10 artikel pertama ───")
        cols = ["date", "source", "kecamatan", "categories", "title"]
        print(df[cols].head(10).to_string(index=False))

    print("\nSelesai.")


if __name__ == "__main__":
    main()

GEO-TEXT MINING – KABUPATEN MALANG
Periode : 2026-01-01 s.d. 2026-03-31
Sumber  : 6 portal berita
Output  : /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/C:/content
Total query: 58 | Sumber: 6
Periode: 2026-01-01 s.d. 2026-03-31


🔍 Query: 'pencurian kabupaten malang'
  [Surya Malang] p1: https://suryamalang.tribunnews.com/search?q=pencurian+kabupaten+malang&p
    ⚠ tidak ada artikel, lanjut halaman berikutnya.
  [Detik Jatim] p1: https://www.detik.com/jatim/search/?query=pencurian+kabupaten+malang&sor
    [HTTP 404] https://www.detik.com/jatim/search/?query=pencurian+kabupaten+malang&s
    ⚠ tidak ada artikel, lanjut halaman berikutnya.
  [Antara Jatim] p1: https://www.antaranews.com/search?q=pencurian+kabupaten+malang&from=2026
    ⚠ tidak ada artikel, lanjut halaman berikutnya.
  [Radar Malang] p1: https://radarmalang.jawapos.com/?s=pencurian+kabupaten+malang&paged=1
    ⚠ tidak ada artikel, lanjut halaman berikutnya.
  [Malang Times] p1: https://www.mala